In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r'D:\Datascience\Project\flight-delay-predictor\data\processed')

# ── File 1: EDA summary data (aggregated — small file) ────────────
train = pd.read_parquet(DATA_DIR / 'train_selected.parquet')
train_pre = pd.read_parquet(DATA_DIR / 'train_preprocessed.parquet')

# Day of week delay rates
dow = train.groupby('day_of_week').agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count'),
    delayed_flights=('departure_delayed','sum')
).reset_index()
dow['day_name'] = dow['day_of_week'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})
dow.to_csv(DATA_DIR / 'pbi_dow.csv', index=False)

# Time of day
tod_cols = ['tod_early_morning','tod_afternoon','tod_evening']
available = [c for c in tod_cols if c in train.columns]
def get_tod(row):
    for c in available:
        if row[c]==1: 
            return c.replace('tod_','').replace('_',' ').title()
    return 'Late Night'
train['time_of_day'] = train[available].apply(get_tod, axis=1)
tod = train.groupby('time_of_day').agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count'),
    delayed_flights=('departure_delayed','sum')
).reset_index()
tod.to_csv(DATA_DIR / 'pbi_tod.csv', index=False)

# Day x Time heatmap
train['dow_label'] = train['day_of_week'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'})
dowtod = train.groupby(['dow_label','time_of_day']).agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count')
).reset_index()
dowtod.to_csv(DATA_DIR / 'pbi_dow_tod.csv', index=False)

# Airline rates
airline = train_pre.groupby('carrier_name').agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count'),
    delayed_flights=('departure_delayed','sum')
).reset_index()
airline.to_csv(DATA_DIR / 'pbi_airline.csv', index=False)

# Airport rates
airport = train_pre.groupby('airport_name').agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count'),
    delayed_flights=('departure_delayed','sum')
).reset_index()
airport.to_csv(DATA_DIR / 'pbi_airport.csv', index=False)

# Route delay rate distribution (binned)
route_rates = train.drop_duplicates('route_delay_rate')[['route_delay_rate']].copy()
route_rates['risk_zone'] = pd.cut(route_rates['route_delay_rate'],
    bins=[0,0.15,0.20,0.25,0.30,1.0],
    labels=['Very Low (<15%)','Low (15-20%)','Average (20-25%)','High (25-30%)','Very High (>30%)'])
route_zone = route_rates.groupby('risk_zone').size().reset_index(name='route_count')
route_zone.to_csv(DATA_DIR / 'pbi_route_zones.csv', index=False)

# Severity bins
sev = train.copy()
sev['severity_bin'] = pd.cut(sev['weather_severity'],
    bins=[0,1,2,3,4,5,7,10,25],
    labels=['0-1','1-2','2-3','3-4','4-5','5-7','7-10','10+'])
sev_stats = sev.groupby('severity_bin', observed=True).agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count')
).reset_index()
sev_stats.to_csv(DATA_DIR / 'pbi_severity.csv', index=False)

# Congestion
cong = train.copy()
cong['congestion_label'] = cong['route_congestion'].map({0:'Low',1:'Medium',2:'High'})
cong_stats = cong.groupby('congestion_label').agg(
    delay_rate=('departure_delayed','mean'),
    total_flights=('departure_delayed','count'),
    delayed_flights=('departure_delayed','sum')
).reset_index()
cong_stats.to_csv(DATA_DIR / 'pbi_congestion.csv', index=False)



In [2]:
# ── pbi_models.csv — REAL numbers from modelling report ──────────
models = pd.DataFrame({
    'model_name': [
        'Logistic Regression',
        'Random Forest',
        'Gradient Boosting',
        'LightGBM',
        'XGBoost (Baseline)',
        'XGBoost (Tuned)'
    ],
    'roc_auc': [
        0.6579,
        0.6714,
        0.6733,
        0.6748,
        0.6752,
        0.6754          # best
    ],
    'recall': [
        0.5923,         # highest recall — catches most delays
        0.5501,
        0.5589,
        0.5594,
        0.5595,
        0.5597
    ],
    'precision': [
        0.3384,
        0.3641,
        0.3615,
        0.3633,
        0.3638,
        0.3638
    ],
    'f1_delayed': [
        0.4307,
        0.4382,
        0.4390,
        0.4405,
        0.4409,
        0.4408          # best F1
    ],
    'train_time_sec': [
        25.2,
        213.5,
        627.6,
        15.4,           # fastest
        33.4,
        33.6
    ],
    'delays_caught': [
        185963,         # most caught
        172724,
        175482,
        175616,
        175652,
        175652
    ],
    'delays_missed': [
        128001,
        141240,
        138482,
        138348,
        138312,
        138312
    ],
    'is_best': [
        0, 0, 0, 0, 0, 1   # flag for highlighting in Power BI
    ]
})
models.to_csv(DATA_DIR / 'pbi_models.csv', index=False)
print('pbi_models.csv saved with real numbers.')

pbi_models.csv saved with real numbers.


In [3]:
target = pd.DataFrame({
    'status':['On-Time','Delayed'],
    'count': [int((train['departure_delayed']==0).sum()),
              int((train['departure_delayed']==1).sum())]
})
target['pct'] = target['count'] / target['count'].sum() * 100
target.to_csv(DATA_DIR / 'pbi_target.csv', index=False)

print('All CSV files exported successfully.')
print('Files saved to:', DATA_DIR)

All CSV files exported successfully.
Files saved to: D:\Datascience\Project\flight-delay-predictor\data\processed
